# Sprint 2: Reentrenamiento con 40 epochs

Las curvas del entrenamiento anterior muestran que el modelo **no ha convergido**:
- Loss train y val bajan juntas y en paralelo (sin overfitting).
- AUC macro sigue subiendo en epoch 20.

Se extiende a 40 epochs con patience=8.

**Pre-requisitos (deben existir en /content/):**
- `/content/nih_images.h5` — HDF5 con 34,999 imágenes.
- `/content/processed_s2/` — CSVs con splits de 5 tarballs (739/148/148).

Si no están en /content/, las celdas 2 y 3 las copian desde Drive.

**Runtime: T4 GPU.**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'
CODE_DIR     = f'{PROJECT_ROOT}/code'

import os, sys, shutil, time, subprocess
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"torch: {torch.__version__}")


## 2. Verificar HDF5 en disco local

In [ ]:
H5_LOCAL = '/content/nih_images.h5'
H5_DRIVE = f'{NIH_DIR}/nih_images.h5'

if os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 1_000_000_000:
    size_gb = os.path.getsize(H5_LOCAL) / (1024**3)
    print(f'✓ HDF5 ya en disco local ({size_gb:.2f} GB).')
else:
    size_gb = os.path.getsize(H5_DRIVE) / (1024**3)
    print(f'Copiando HDF5 ({size_gb:.2f} GB) de Drive a /content/...')
    t0 = time.time()
    shutil.copy2(H5_DRIVE, H5_LOCAL)
    elapsed = time.time() - t0
    print(f'✓ Copiado en {elapsed/60:.1f} min')

import h5py
with h5py.File(H5_LOCAL, 'r') as hf:
    n = len(hf['images'])
print(f'Imágenes en HDF5: {n:,}')


## 3. Verificar CSVs de splits

In [ ]:
import pandas as pd

PROCESSED_DIR = '/content/processed_s2'
os.makedirs(PROCESSED_DIR, exist_ok=True)

CLASS_NAMES = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}

for csv in ['train.csv', 'val.csv', 'test.csv']:
    local = f'{PROCESSED_DIR}/{csv}'
    drive = f'{PROJECT_ROOT}/data/processed_s2/{csv}'

    # Copiar desde Drive si no existe localmente o está desactualizado
    if not os.path.exists(local):
        shutil.copy(drive, local)
        print(f'Copiado: {csv}')

    df = pd.read_csv(local)
    dist = {CLASS_NAMES[l]: int((df['label']==l).sum()) for l in range(4)}
    print(f'{csv}: {len(df)} imgs | {dist}')

print()
# Verificar que son los splits de 5 tarballs (739/148/148)
train_n = len(pd.read_csv(f'{PROCESSED_DIR}/train.csv'))
if train_n < 700:
    print('⚠ Train tiene menos de 700 imgs — puede ser splits de 3 tarballs.')
    print('  Copiando splits actualizados desde Drive...')
    for csv in ['train.csv', 'val.csv', 'test.csv']:
        shutil.copy(f'{PROJECT_ROOT}/data/processed_s2/{csv}',
                    f'{PROCESSED_DIR}/{csv}')
    print('✓ Splits actualizados.')
else:
    print('✓ Splits correctos (5 tarballs).')


## 4. Instalar dependencias

In [ ]:
!pip install -q torchxrayvision scipy h5py
print('deps OK')


## 5. Actualizar `sprint2.yaml`: epochs 20 → 40, patience 5 → 8

In [ ]:
yaml_path = f'{CODE_DIR}/configs/sprint2.yaml'

with open(yaml_path, encoding='utf-8') as f:
    content = f.read()

# Mostrar valores actuales
import re
epochs_match  = re.search(r'epochs:\s*(\d+)', content)
patience_match = re.search(r'patience:\s*(\d+)', content)
print(f'Valores actuales:')
print(f'  epochs:  {epochs_match.group(1) if epochs_match else "?"}')
print(f'  patience: {patience_match.group(1) if patience_match else "?"}')

# Actualizar
content = re.sub(r'epochs:\s*\d+', 'epochs: 40', content)
content = re.sub(r'patience:\s*\d+', 'patience: 8', content)

with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(content)

# Verificar
with open(yaml_path) as f:
    updated = f.read()
e = re.search(r'epochs:\s*(\d+)', updated)
p = re.search(r'patience:\s*(\d+)', updated)
print(f'\nValores actualizados:')
print(f'  epochs:   {e.group(1) if e else "?"}')
print(f'  patience: {p.group(1) if p else "?"}')
print('✓ YAML actualizado')


## 6. Smoke test: verificar que el dataset carga bien

In [ ]:
# Limpiar cache de imports
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'): del sys.modules[mod]

from src.datasets.nih_hdf5 import build_nih_hdf5_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.training.utils import load_config

cfg = load_config(f'{CODE_DIR}/configs/sprint2.yaml')

train_ds, val_ds, test_ds = build_nih_hdf5_datasets(
    PROCESSED_DIR, H5_LOCAL,
    build_train_transform(224),
    build_eval_transform(224),
)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Distribución train: {train_ds.class_distribution()}')

# Cargar 1 muestra para verificar
sample = train_ds[0]
print(f'\nMuestra[0]: shape={sample["image"].shape} | '
      f'range=[{sample["image"].min():.0f}, {sample["image"].max():.0f}] | '
      f'label={sample["label"].item()} ({CLASS_NAMES[sample["label"].item()]})')
print('✓ Dataset OK. Listo para entrenar.')


## 7. Entrenar con 40 epochs

Tiempo estimado: **~60 min** en T4 (el doble que antes, 40 epochs en vez de 20).

Verás un log por epoch. Las métricas clave:
- `AUC_macro` en val → queremos superar 0.82.
- `val AUC clase 3` (Infiltration) → esperamos llegar a 0.75+.
- Loss train y val deben seguir bajando juntas.


In [ ]:
env = {
    **os.environ,
    'H5_PATH':       H5_LOCAL,
    'PROCESSED_DIR': PROCESSED_DIR,
}

print('Iniciando entrenamiento Sprint 2 (40 epochs)...')
print('Esto tarda ~60 min en T4. No cierres la pestaña.')
print()

result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/run_train_s2_hdf5.py'],
    capture_output=True, text=True, env=env,
)

print('STDOUT:')
print(result.stdout[-1000:] if result.stdout else '(vacío)')
print()
print('LOG:')
# Mostrar las últimas 80 líneas del log
lines = result.stderr.strip().split('\n') if result.stderr else []
print('\n'.join(lines[-80:]))
print()
print(f'Return code: {result.returncode}')


## 8. Curvas de entrenamiento y métricas finales

In [ ]:
import json, matplotlib.pyplot as plt

history_path = f'{PROJECT_ROOT}/experiments_s2/history_s2.json'

if not os.path.exists(history_path):
    print('history_s2.json no encontrado. Revisa la celda anterior.')
else:
    with open(history_path) as f:
        data = json.load(f)

    history    = data['history']
    epochs     = [h['epoch'] for h in history]
    CLASS_NAMES = {0:'No Finding', 1:'Cardiomegaly',
                   2:'Effusion',   3:'Infiltration'}

    print(f"Best epoch: {data['best_epoch']} | "
          f"AUC_macro: {data['best_val_auc_macro']:.4f}")
    print()

    best_h = history[data['best_epoch'] - 1]
    print('Métricas en el mejor epoch (val):')
    print(f"  AUC macro:   {best_h['val_metrics']['auc_macro']:.4f}")
    print(f"  Accuracy:    {best_h['val_metrics']['accuracy']:.4f}")
    print(f"  Prec macro:  {best_h['val_metrics']['precision_macro']:.4f}")
    print(f"  Sens macro:  {best_h['val_metrics']['sensitivity_macro']:.4f}")
    print(f"  Spec macro:  {best_h['val_metrics']['specificity_macro']:.4f}")
    print()
    for i, auc in enumerate(best_h['val_metrics']['auc_per_class']):
        sens = best_h['val_metrics']['sensitivity_per_class'][i]
        spec = best_h['val_metrics']['specificity_per_class'][i]
        print(f"  {CLASS_NAMES[i]:<15}: AUC={auc:.4f} | Sens={sens:.4f} | Spec={spec:.4f}")

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']

    # Loss
    axes[0].plot(epochs, [h['train_loss'] for h in history],
                 'o-', ms=3, label='train', color='steelblue')
    axes[0].plot(epochs, [h['val_loss'] for h in history],
                 's-', ms=3, label='val', color='coral')
    axes[0].axvline(data['best_epoch'], color='gray', linestyle=':', alpha=0.7)
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    # AUC macro
    axes[1].plot(epochs, [h['train_metrics']['auc_macro'] for h in history],
                 'o-', ms=3, label='train', color='steelblue')
    axes[1].plot(epochs, [h['val_metrics']['auc_macro'] for h in history],
                 's-', ms=3, label='val', color='coral')
    axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.4, label='random')
    axes[1].axvline(data['best_epoch'], color='gray', linestyle=':', alpha=0.7)
    axes[1].set_title('AUC Macro'); axes[1].set_ylim(0, 1.05)
    axes[1].legend(); axes[1].grid(alpha=0.3)

    # AUC por clase
    for i in range(4):
        axes[2].plot(epochs,
                     [h['val_metrics']['auc_per_class'][i] for h in history],
                     'o-', ms=3, label=CLASS_NAMES[i], color=colors[i])
    axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.4)
    axes[2].axvline(data['best_epoch'], color='gray', linestyle=':', alpha=0.7,
                    label=f'best epoch={data["best_epoch"]}')
    axes[2].set_title('Val AUC por clase'); axes[2].set_ylim(0, 1.05)
    axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

    for ax in axes:
        ax.set_xlabel('Epoch')

    plt.suptitle(
        f"Sprint 2 (40 epochs) | AUC_macro={data['best_val_auc_macro']:.4f} "
        f"@ epoch {data['best_epoch']}"
    )
    plt.tight_layout()
    os.makedirs(f'{PROJECT_ROOT}/experiments_s2/figures', exist_ok=True)
    plt.savefig(
        f'{PROJECT_ROOT}/experiments_s2/figures/curves_s2_40ep.png',
        dpi=120, bbox_inches='tight'
    )
    plt.show()
    print('✓ Gráfica guardada.')


## 9. Comparativa Sprint 1 vs Sprint 2

In [ ]:
# Cargar métricas de Sprint 1
import json, os

s1_path = f'{PROJECT_ROOT}/experiments/test_metrics.json'
s2_path = f'{PROJECT_ROOT}/experiments_s2/history_s2.json'

if not os.path.exists(s1_path) or not os.path.exists(s2_path):
    print('Faltan archivos de métricas. Revisa las rutas.')
else:
    with open(s1_path) as f:
        s1 = json.load(f)['metrics']
    with open(s2_path) as f:
        s2_data = json.load(f)
    s2_best = s2_data['history'][s2_data['best_epoch'] - 1]['val_metrics']

    print('=' * 65)
    print('COMPARATIVA SPRINT 1 vs SPRINT 2')
    print('=' * 65)
    print(f"{'Métrica':<20} {'Sprint 1':>12} {'Sprint 2':>12} {'Delta':>10}")
    print('-' * 65)

    # Sprint 1 (binario): AUC sobre test
    # Sprint 2 (4 clases): AUC macro sobre val (test pendiente)
    comparisons = [
        ('AUC (macro/global)', s1['auc'],      s2_best['auc_macro']),
        ('Accuracy',           s1['accuracy'],  s2_best['accuracy']),
        ('Precision macro',    s1['precision'], s2_best['precision_macro']),
        ('Sensitivity macro',  s1['sensitivity'],s2_best['sensitivity_macro']),
        ('Specificity macro',  s1['specificity'],s2_best['specificity_macro']),
    ]

    for name, v1, v2 in comparisons:
        delta = v2 - v1
        arrow = '↑' if delta > 0.02 else ('↓' if delta < -0.02 else '≈')
        print(f"{name:<20} {v1:>12.4f} {v2:>12.4f} {delta:>+10.4f} {arrow}")

    print()
    print('Notas:')
    print('  Sprint 1: 2 clases (binario), 199 imgs train, finetune HEAD.')
    print('  Sprint 2: 4 clases (multi-clase), 739 imgs train, finetune FULL.')
    print('  Los AUC no son directamente comparables (distinta dificultad).')
    print('  Sprint 2 incluye Infiltration (TB/neumonía) — clave para HNAL.')
    print()
    print('AUC por clase Sprint 2:')
    for i, auc in enumerate(s2_best['auc_per_class']):
        print(f'  {CLASS_NAMES[i]:<15}: {auc:.4f}')


In [ ]:
# Celda rápida: evaluar el mejor checkpoint sobre test
import subprocess, os

env = {**os.environ, 'H5_PATH': H5_LOCAL, 'PROCESSED_DIR': PROCESSED_DIR}

result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/run_eval_s2.py'],
    capture_output=True, text=True, env=env,
)
print(result.stderr[-3000:])

## 10. Siguiente paso

Si AUC macro supera 0.82 → **Sprint 2 cerrado**.

Vuelve al chat para recibir:
1. `evaluate_s2.py` — evaluación final sobre test set (métricas definitivas).
2. `gradcam_s2.py` — Grad-CAM para 4 clases (visualización por clase).
3. Documentación del Sprint 2.
4. Plan del Sprint 4: modelo híbrido CNN-ViT.
